## Ingest lap_times directory csv


### Step 0 - Parameters and Imports


In [0]:
%run "../utils/config"


In [0]:
dbutils.widgets.text("p_source_value", "")
v_source_value = dbutils.widgets.get("p_source_value")


In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType, DateType, TimestampType, FloatType
from pyspark.sql.functions import col, lit, current_timestamp, to_timestamp, concat

### Step 1 - Read the csv file


#### 1.1 Define the schema


In [0]:
laptimes_schema = StructType(fields=[
    StructField("raceId",IntegerType(),False),
    StructField("driverId",IntegerType(),True),
    StructField("lap",IntegerType(),True),
    StructField("position",IntegerType(),True),
    StructField("time",StringType(),True),
    StructField("milliseconds",IntegerType(),True),
])


#### 1.2 Read the csv file


In [0]:
raw_path = raw_folder_path


In [0]:
laptimes_df = (
    spark.read
        .format("csv")
        .schema(laptimes_schema)
        .load(f"{raw_path}/lap_times")
)


### Step 2 - Transform the data


In [0]:
renamed_laptimes_df = add_ingestion_date(
    laptimes_df
        .withColumnRenamed("raceId", "race_id")
        .withColumnRenamed("driverId", "driver_id")
        .withColumn("source", lit(v_source_value)))


### Step 3 - Write the output to parquet


In [0]:
processed_path = processed_folder_path


In [0]:
renamed_laptimes_df.write.format("parquet").mode("overwrite").save(f"{processed_path}/lap_times")


In [0]:
dbutils.notebook.exit("success")